# Where is the bug? Testing file-finding methods on the 129 public tasks

Every coding agent in this competition starts the same way: **read the issue, then find the file to edit.**
If it opens the wrong file, the rest of the budget is wasted.

The public tasks ship with the reference fix (`patch`), so we know exactly which files a correct fix touches.
That gives us a clean, cheap test: **for each search method, how often is a gold file in its top k?**

Methods compared (all CPU, no Gemma, no GPU quota):

| # | Method | Idea |
|---|---|---|
| 1 | Path match | issue words that appear in the file path |
| 2 | Grep identifiers | code-looking words from the issue (`snake_case`, `CamelCase`, `dotted.names`) found in the file |
| 3 | BM25 on files | classic text search over whole files |
| 4 | BM25 on graph nodes | text search over functions/classes from `graphs/`, best node per file |
| 5 | Graph expansion | method 4, plus the call-graph neighbours of the top nodes |
| 6 | Embedding expansion | method 4, plus nodes close in the provided `embeddings/` space |
| 7 | Hybrid (RRF) | reciprocal-rank fusion of methods 2, 3, 4, 6 |

Metric: **Recall@k** = share of tasks where at least one gold (non-test) file is in the top k. Also MRR.


In [ ]:
# all imports in one place
import os, re, json, glob, math, tarfile
from collections import Counter, defaultdict
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option('display.width', 160)

# competition data (found wherever Kaggle mounts it).
# There can be more than one tasks.jsonl: we keep the one that carries the reference `patch`.
def has_patch(path):
    t = pd.read_json(path, lines=True)
    return 'patch' in t and t.patch.astype(str).str.contains(r'diff --git|\+\+\+ b/').any()

cands = [os.environ['G4_TASKS']] if os.environ.get('G4_TASKS') else sorted(glob.glob('/kaggle/input/**/tasks.jsonl', recursive=True))
TASKS = next((p for p in cands if has_patch(p)), cands[0])
tasks = pd.read_json(TASKS, lines=True)

# snapshots/, graphs/ and embeddings/ sit next to each other; find that folder
DATA = next((os.path.dirname(d) for d in glob.glob(os.path.dirname(TASKS) + '/../**/snapshots', recursive=True)), None) \
    or next((os.path.dirname(d) for d in glob.glob('/kaggle/input/**/snapshots', recursive=True)), os.path.dirname(TASKS))
DATA = os.path.normpath(DATA)
print('tasks from:', TASKS, '| data from:', DATA)
print(len(tasks), 'tasks |', tasks.repo.value_counts().to_dict())
print('columns:', list(tasks.columns))
print('patch sample:', repr(str(tasks['patch'].iloc[0])[:300]) if 'patch' in tasks else 'no patch column')


## 1. Ground truth: which files does the reference fix touch?

We parse the `diff --git` headers of each `patch` and keep `.py` files that are not tests.
Test files live in `test_patch`, so they are excluded by construction, but we filter again to be safe.


In [ ]:
def patch_files(patch):
    """Files changed by a unified diff."""
    if not isinstance(patch, str):
        patch = '' if patch is None or (isinstance(patch, float) and math.isnan(patch)) else json.dumps(patch)
    if '\n' not in patch and '\\n' in patch:   # double-escaped text: turn literal \n into real newlines
        patch = patch.replace('\\n', '\n')
    patch = patch.replace('\r\n', '\n')
    files = re.findall(r'^diff --git a/(\S+) b/', patch, flags=re.M)
    files += re.findall(r'^\+\+\+ b/(\S+)', patch, flags=re.M)   # also plain '--- a/ +++ b/' diffs
    return list(dict.fromkeys(files))

def is_test(path):
    """True for test files and test folders."""
    return bool(re.search(r'(^|/)(tests?|testing)/|(^|/)test_[^/]*\.py$|_test\.py$|conftest\.py$', path))

def changed_lines(patch):
    """Added + removed lines in a diff (not counting headers)."""
    return sum(1 for l in (patch or '').splitlines() if l[:1] in '+-' and not l.startswith(('+++', '---')))

tasks['gold'] = tasks.patch.map(lambda p: [f for f in patch_files(p) if f.endswith('.py') and not is_test(f)])
tasks['n_gold'] = tasks.gold.map(len)
tasks['lines'] = tasks.patch.map(changed_lines)
tasks = tasks[tasks.n_gold > 0].reset_index(drop=True)

summary = tasks.groupby('repo').agg(tasks=('instance_id', 'size'),
                                     files_per_fix=('n_gold', 'mean'),
                                     lines_per_fix=('lines', 'median'))
print(summary.round(2))
print(f"\n{(tasks.n_gold == 1).mean():.0%} of fixes touch a single source file")


## 2. Load each repository snapshot and its code graph

* `snapshots/<id>.tgz` gives every `.py` file at `base_commit` (we keep non-test source files as candidates).
* `graphs/<id>.json` gives functions/classes with their source text and call edges.
* `embeddings/<id>.npz` gives a 256-d vector per graph node.

Graph node ids are dotted Python paths (`fastapi.routing.APIRoute`), so we map each node to a file by the longest
module prefix that matches a candidate file.


In [ ]:
def load_repo(iid, gold):
    """Return {relative_path: source} for non-test .py files in the snapshot."""
    tf = tarfile.open(f'{DATA}/snapshots/{iid}.tgz')
    members = {m.name: m for m in tf.getmembers() if m.isfile() and m.name.endswith('.py') and '/.git/' not in m.name}
    # the archive may wrap the repo in a top folder: find the prefix that makes gold paths line up
    prefix = ''
    for name in members:
        hit = next((g for g in gold if name == g or name.endswith('/' + g)), None)
        if hit:
            prefix = name[: len(name) - len(hit)]
            break
    files = {}
    for name, m in members.items():
        rel = name[len(prefix):] if name.startswith(prefix) else name
        rel = rel[2:] if rel.startswith('./') else rel
        if not is_test(rel):
            files[rel] = tf.extractfile(m).read().decode('utf-8', 'ignore')
    return files

def module_map(files):
    """'fastapi/routing.py' -> 'fastapi.routing' (also strips a src/ layout)."""
    out = {}
    for f in files:
        mod = f[:-3].replace('/', '.')
        mod = mod[:-9] if mod.endswith('.__init__') else mod
        out[mod] = f
        if mod.startswith('src.'):
            out[mod[4:]] = f
    return out

def node_file(node_id, mods):
    """Longest dotted prefix of a node id that is a known module."""
    parts = node_id.split('.')
    for k in range(len(parts), 0, -1):
        f = mods.get('.'.join(parts[:k]))
        if f:
            return f
    return None

# Graph and embedding files exist under two names: <instance_id> and <repo_short>_<base_commit>.
# Only one of them may be present in the Kaggle copy, so we look for either.
ALL_FILES = glob.glob(f'{DATA}/**/*.json', recursive=True) + glob.glob(f'{DATA}/**/*.npz', recursive=True)

def find_file(row, folder, ext):
    names = {f'{row.instance_id}{ext}', f"{row.repo.split('/')[-1].lower()}_{row.base_commit}{ext}"}
    hits = [p for p in ALL_FILES if f'/{folder}/' in p and os.path.basename(p) in names]
    hits = hits or [p for p in ALL_FILES if f'/{folder}/' in p and p.endswith(ext) and row.base_commit in os.path.basename(p)]
    return hits[0] if hits else None

print(sum('/graphs/' in p for p in ALL_FILES), 'graph files |', sum('/embeddings/' in p for p in ALL_FILES), 'embedding files |',
      'graph found for', sum(find_file(r, 'graphs', '.json') is not None for r in tasks.itertuples()), 'of', len(tasks), 'tasks')

def load_graph(row, mods):
    path = find_file(row, 'graphs', '.json')
    if path is None:
        return [], [], [], {}
    g = json.load(open(path))
    nodes = [n for n in g['nodes'] if n.get('text')]
    nid = [n['id'] for n in nodes]
    nfile = [node_file(i, mods) for i in nid]
    keep = [k for k, f in enumerate(nfile) if f]
    nodes, nid, nfile = [nodes[k] for k in keep], [nid[k] for k in keep], [nfile[k] for k in keep]
    index = {i: k for k, i in enumerate(nid)}
    nbrs = defaultdict(set)
    for e in g.get('edges', g.get('links', [])):
        a, b = index.get(e.get('source')), index.get(e.get('target'))
        if a is not None and b is not None and a != b:
            nbrs[a].add(b); nbrs[b].add(a)
    return nodes, nid, nfile, nbrs

def load_emb(row, nid):
    """Matrix of unit vectors aligned with nid (zeros where a node has no vector)."""
    path = find_file(row, 'embeddings', '.npz')
    if path is None:
        return np.zeros((len(nid), 256), dtype=np.float32)
    z = np.load(path)
    keys = {k[:-4] if k.endswith('.npy') else k: k for k in z.files}
    E = np.zeros((len(nid), 256), dtype=np.float32)
    for k, i in enumerate(nid):
        if i in keys:
            v = np.asarray(z[keys[i]], dtype=np.float32).ravel()[:256]
            E[k, :len(v)] = v
    norm = np.linalg.norm(E, axis=1, keepdims=True)
    return E / np.where(norm == 0, 1, norm)


## 3. Text tools: tokenizer and BM25

The tokenizer keeps whole identifiers **and** their pieces (`get_response_content` -> `get`, `response`, `content`),
so an issue that says "response content" still matches the function name.


In [ ]:
STOP = set('the a an and or of to in is it this that for on with as be by are was were not no if when then than from at '
           'i we you they he she my our your can could should would will do does did have has had but so use using used '
           'get set self none true false return def class import python error issue bug fix expected actual '
           'example code like also one two any all just some more'.split())

def tokens(text):
    out = []
    for w in re.findall(r'[A-Za-z_][A-Za-z0-9_]*', text or ''):
        parts = [p for p in re.split(r'_|(?<=[a-z0-9])(?=[A-Z])', w) if p]
        for t in {w.lower(), *[p.lower() for p in parts]}:
            if len(t) > 2 and t not in STOP:
                out.append(t)
    return out

def code_words(text):
    """Words in the issue that look like code: snake_case, CamelCase, dotted.names, `backticked`."""
    words = set(re.findall(r'`([^`\n]{2,60})`', text or ''))
    words |= set(re.findall(r'\b[A-Za-z_]+(?:\.[A-Za-z_]+)+\b', text or ''))
    words |= set(re.findall(r'\b[a-z]+_[a-z0-9_]+\b', text or ''))
    words |= set(re.findall(r'\b[A-Z][a-z]+(?:[A-Z][a-z0-9]+)+\b', text or ''))
    out = set()
    for w in words:
        out |= {p for p in re.split(r'[^A-Za-z0-9_]+', w) if len(p) > 2}
    return out

class BM25:
    """Okapi BM25 with an inverted index (fast enough for thousands of nodes)."""
    def __init__(self, docs, k1=1.2, b=0.75):
        self.n = len(docs)
        lens = np.array([len(d) for d in docs], dtype=float)
        self.norm = k1 * (1 - b + b * lens / (lens.mean() if self.n and lens.mean() > 0 else 1))
        self.k1 = k1
        self.post = defaultdict(list)
        for i, d in enumerate(docs):
            for t, c in Counter(d).items():
                self.post[t].append((i, c))
        self.idf = {t: math.log(1 + (self.n - len(p) + .5) / (len(p) + .5)) for t, p in self.post.items()}

    def score(self, query):
        s = np.zeros(self.n)
        for t in set(query):
            for i, c in self.post.get(t, ()):
                s[i] += self.idf[t] * c * (self.k1 + 1) / (c + self.norm[i])
        return s


## 4. The seven methods

Each method returns a score per candidate file; files are ranked by score.
Node-level methods give a file the score of its best node.


In [ ]:
def per_file(node_scores, nfile, files):
    best = defaultdict(float)
    for s, f in zip(node_scores, nfile):
        if s > best[f]:
            best[f] = s
    return np.array([best.get(f, 0.0) for f in files])

def rank_of(scores, files, gold):
    """1-based rank of the first gold file (ties broken by file order); inf if no gold among candidates."""
    order = np.argsort(-scores, kind='stable')
    for r, k in enumerate(order, 1):
        if files[k] in gold and scores[k] > 0:
            return r
    return np.inf

def rrf(*score_lists, k=60):
    """Reciprocal-rank fusion: sum of 1 / (k + rank) across methods."""
    out = np.zeros(len(score_lists[0]))
    for s in score_lists:
        ranks = np.empty(len(s)); ranks[np.argsort(-s, kind='stable')] = np.arange(1, len(s) + 1)
        out += np.where(s > 0, 1 / (k + ranks), 0)
    return out

def run_task(row):
    gold = set(row.gold)
    src = load_repo(row.instance_id, row.gold)
    files = sorted(src)
    if not gold & set(files):
        return None
    mods = module_map(files)
    query_text = row.problem_statement
    q = tokens(query_text)
    cw = {w.lower() for w in code_words(query_text)}

    s = {}
    # 1. path match: issue tokens that appear in the path
    qs = set(q)
    s['1 path match'] = np.array([len(qs & set(tokens(f.replace('/', ' ')))) for f in files], float)
    # 2. grep code-looking words
    low = [src[f].lower() for f in files]
    s['2 grep identifiers'] = np.array([sum(w in t for w in cw) for t in low], float)
    # 3. BM25 over whole files
    s['3 BM25 files'] = BM25([tokens(src[f]) + tokens(f.replace('/', ' ')) for f in files]).score(q)

    # graph-based methods
    nodes, nid, nfile, nbrs = load_graph(row, mods)
    if nodes:
        node_bm = BM25([tokens(n['text']) + tokens(n['id']) for n in nodes]).score(q)
        s['4 BM25 graph nodes'] = per_file(node_bm, nfile, files)
        # 5. add call-graph neighbours of the top 10 nodes (half weight)
        top = np.argsort(-node_bm)[:10]
        g_scores = node_bm.copy()
        for t in top:
            for nb in nbrs.get(t, ()):
                g_scores[nb] = max(g_scores[nb], 0.5 * node_bm[t])
        s['5 graph expansion'] = per_file(g_scores, nfile, files)
        # 6. embedding expansion: nodes similar to the centroid of the top 5 BM25 nodes
        E = load_emb(row, nid)
        if np.abs(E).sum() > 0:
            c = E[np.argsort(-node_bm)[:5]].mean(0)
            sim = np.clip(E @ (c / (np.linalg.norm(c) or 1)), 0, None)
            e_scores = node_bm / (node_bm.max() or 1) + 0.5 * sim
            s['6 embedding expansion'] = per_file(e_scores, nfile, files)
    parts = [s[k] for k in ('2 grep identifiers', '3 BM25 files', '4 BM25 graph nodes', '6 embedding expansion') if k in s]
    s['7 hybrid (RRF)'] = rrf(*parts)

    out = {'instance_id': row.instance_id, 'repo': row.repo, 'n_files': len(files), 'n_nodes': len(nodes)}
    out.update({m: rank_of(v, files, gold) for m, v in s.items()})
    return out


In [ ]:
import time
t0 = time.time()
rows = []
for i, row in enumerate(tasks.itertuples()):
    try:
        r = run_task(row)
        if r:
            rows.append(r)
    except Exception as e:
        print('skip', row.instance_id, type(e).__name__, e)
    if i % 20 == 0:
        print(f'{i}/{len(tasks)}  {time.time() - t0:.0f}s')
res = pd.DataFrame(rows)
METHODS = [c for c in res.columns if c[0].isdigit()]
print(f'{len(res)} tasks scored in {time.time() - t0:.0f}s | median {res.n_files.median():.0f} candidate files per repo')


## 5. Results: how often is the right file in the top k?


In [ ]:
def recall_table(df):
    rows = {}
    for m in METHODS:
        r = df[m].fillna(np.inf)
        rows[m] = {'R@1': (r <= 1).mean(), 'R@3': (r <= 3).mean(), 'R@5': (r <= 5).mean(),
                   'R@10': (r <= 10).mean(), 'MRR': (1 / r).mean()}
    return pd.DataFrame(rows).T

table = recall_table(res)
print(table.round(3))

fig, ax = plt.subplots(figsize=(10, 4.5))
table[['R@1', 'R@5', 'R@10']].plot.barh(ax=ax, color=['#1f4e79', '#4c8bc4', '#a9c8e8'])
ax.invert_yaxis(); ax.set_xlim(0, 1); ax.set_xlabel('share of tasks with a gold file in the top k')
ax.set_title('File localisation on the 129 public tasks'); ax.legend(loc='lower right')
plt.tight_layout(); plt.show()


## 6. By repository

Some repos are much harder: large packages with many similar files dilute text search.


In [ ]:
by_repo = {repo: recall_table(g)['R@5'] for repo, g in res.groupby('repo')}
by_repo = pd.DataFrame(by_repo)
print('Recall@5 by repo'); print(by_repo.round(2))

fig, ax = plt.subplots(figsize=(8, 4))
im = ax.imshow(by_repo.values, cmap='Blues', vmin=0, vmax=1, aspect='auto')
ax.set_xticks(range(by_repo.shape[1])); ax.set_xticklabels(by_repo.columns, rotation=20)
ax.set_yticks(range(by_repo.shape[0])); ax.set_yticklabels(by_repo.index)
for (i, j), v in np.ndenumerate(by_repo.values):
    ax.text(j, i, f'{v:.2f}', ha='center', va='center', color='white' if v > .6 else 'black', fontsize=8)
ax.set_title('Recall@5 by repository'); plt.colorbar(im, ax=ax); plt.tight_layout(); plt.show()


## 7. Where do the misses come from?

Tasks where even the best method misses the gold file in its top 10 are the ones an agent must solve by
exploring (running tests, reading tracebacks), not by search.


In [ ]:
best = table['R@10'].idxmax()
miss = res[res[best].fillna(np.inf) > 10][['instance_id', 'repo', 'n_files', best]]
print(f'best method by R@10: {best} | misses: {len(miss)} of {len(res)}')
print(miss.head(15).to_string(index=False))

fig, ax = plt.subplots(figsize=(8, 3.5))
r = res[best].replace(np.inf, np.nan)
ax.hist(r.dropna().clip(upper=30), bins=30, color='#4c8bc4')
ax.set_xlabel('rank of the first gold file (capped at 30)'); ax.set_ylabel('tasks')
ax.set_title(f'Rank distribution, {best}'); plt.tight_layout(); plt.show()


## 8. Takeaways for agent builders

* Read the table in section 5 from the top: the method with the best **R@5** is the one to put in your agent's first
  step (or in its system prompt as "search like this first").
* Give the agent the **top 5 files**, not the top 1: the jump from R@1 to R@5 is the cheapest accuracy you can buy.
* Use the misses in section 7 as a small hard set to test your exploration loop (tests + tracebacks).
* Everything here is CPU-only and runs in a couple of minutes, so it is cheap to re-run as you change the query
  (e.g. adding `hints_text`) or the tokenizer.

*Data: the competition's public tasks, snapshots, graphs and embeddings (Apache 2.0). Method ideas are standard IR:
BM25 (Robertson & Zaragoza) and reciprocal-rank fusion (Cormack et al., 2009).*
